In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression


In [3]:
## Loading the data
train_df = pd.read_csv('Train.csv')
test_df = pd.read_csv('Test.csv')
#y_true = pd.read_csv('reference.csv')['Target_AUC']

train_df_ = pd.read_csv('Train_.csv')
test_df_ = pd.read_csv('Test_.csv')
#y_true_ = pd.read_csv('reference_.csv')['Target_AUC']

In [5]:
for c in [col for col in train_df.columns if col.endswith('date')]:
  if c in train_df.columns:
    train_df[c] = pd.to_datetime(train_df[c], errors='coerce')


for c in [col for col in train_df_.columns if col.endswith('date')]:
  if c in train_df_.columns:
    train_df_[c] = pd.to_datetime(train_df_[c], errors='coerce')


In [6]:
target_col = 'adopted_within_07_days'
train_df[target_col] = train_df[target_col].astype(int)

target_col = 'adopted_within_07_days'
train_df_[target_col] = train_df_[target_col].astype(int)

In [7]:
for c in [col for col in test_df.columns if col.endswith('date')]:
  if c in test_df.columns:
    test_df[c] = pd.to_datetime(test_df[c], errors='coerce')


for c in [col for col in test_df_.columns if col.endswith('date')]:
  if c in test_df_.columns:
    test_df_[c] = pd.to_datetime(test_df_[c], errors='coerce')

In [8]:
split_summary = pd.DataFrame({
    "set": ["train"],
    "rows": [len(train_df)],
    "positives": [train_df[target_col].sum()],
})
split_summary["pos_rate"] = split_summary["positives"] / split_summary["rows"]

split_summary

,set,rows,positives,pos_rate
0,train,13536,153,0.011303


In [9]:
split_summary_ = pd.DataFrame({
    "set": ["train"],
    "rows": [len(train_df_)],
    "positives": [train_df_[target_col].sum()],
})
split_summary_["pos_rate"] = split_summary_["positives"] / split_summary_["rows"]

split_summary_

,set,rows,positives,pos_rate
0,train,13536,153,0.011303


In [17]:
## Basic model with some selected features
base_features = [
    "gender",
    "registration",
    "age",
    "trainer",
    "belong_to_cooperative",
    "county",
    "subcounty",
    "ward",
]

In [11]:
feature_cols = base_features

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test  = test_df[feature_cols]


X_train_ = train_df_[feature_cols]
y_train_ = train_df_[target_col]

X_test_  = test_df_[feature_cols]

## Make the prediction


In [12]:
preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]), feature_cols)
    ]
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))
])

model.fit(X_train, y_train)
y_pred_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

In [13]:
preprocess_ = ColumnTransformer(
    transformers=[
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]), feature_cols)
    ]
)

model_ = Pipeline(steps=[
    ("preprocess", preprocess_),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))
])

model_.fit(X_train_, y_train_)
y_pred_prob_ = model.predict_proba(X_test_)[:, 1]
y_pred_ = model_.predict(X_test_)

In [14]:
y_pred

array([0, 0, 1, ..., 0, 0, 0])

In [15]:
## Evaluating some metrics
# ----------------------------
#  Metrics: PR AUC + ROC AUC + Recall@K
# ----------------------------
def recall_at_k(y_true: pd.Series, y_scores: np.ndarray, k_frac: float) -> float:
    k = int(np.ceil(len(y_true) * k_frac))
    order = np.argsort(-y_scores)  # descending
    topk = order[:k]
    return float(y_true.iloc[topk].sum() / y_true.sum()) if y_true.sum() > 0 else np.nan

In [16]:
pr_auc = average_precision_score(y_true, y_pred_prob)
roc_auc = roc_auc_score(y_true, y_pred_prob)
recall_5 = recall_at_k(y_true, y_pred_prob, 0.05)
recall_10 = recall_at_k(y_true, y_pred_prob, 0.10)
recall_20 = recall_at_k(y_true, y_pred_prob, 0.20)

# Naive baselines on this test set
prevalence_test = y_true.mean()
naive_pr_auc = prevalence_test

results = pd.DataFrame({
    "metric": [
        "PR AUC (Average Precision)",
        "ROC AUC",
        "Recall@5%",
        "Recall@10%",
        "Recall@20%",
        "Naive PR AUC (prevalence)",
    ],
    "value": [pr_auc, roc_auc, recall_5, recall_10, recall_20, naive_pr_auc]
})


print("\nSplit summary:")
display(split_summary)
print("\nResults:")
display(results)

NameError: name 'y_true' is not defined

In [ ]:
y_pred_prob

In [ ]:
y_pred

In [ ]:
pr_auc_ = average_precision_score(y_true_, y_pred_prob_)
roc_auc_ = roc_auc_score(y_true_, y_pred_prob_)
recall_5_ = recall_at_k(y_true_, y_pred_prob_, 0.05)
recall_10_ = recall_at_k(y_true_, y_pred_prob_, 0.10)
recall_20_ = recall_at_k(y_true_, y_pred_prob_, 0.20)

# Naive baselines on this test set
prevalence_test_ = y_true_.mean()
naive_pr_auc_ = prevalence_test_

results_ = pd.DataFrame({
    "metric": [
        "PR AUC (Average Precision)",
        "ROC AUC",
        "Recall@5%",
        "Recall@10%",
        "Recall@20%",
        "Naive PR AUC (prevalence)",
    ],
    "value": [pr_auc_, roc_auc_, recall_5_, recall_10_, recall_20_, naive_pr_auc_]
})


print("\nSplit summary:")
display(split_summary_)
print("\nResults:")
display(results_)

In [ ]:
## Make submission file
ss = pd.read_csv('SampleSubmission.csv')
ss['Target_LogLoss'] = y_pred_prob
ss['Target_AUC'] = y_pred_prob
ss.to_csv('BenchmarkSub.csv', index=False)

ss_ = pd.read_csv('SampleSubmission_.csv')
ss_['Target_LogLoss'] = y_pred_prob_
ss_['Target_AUC'] = y_pred_prob_
ss_.to_csv('BenchmarkSub_.csv', index=False)

In [ ]:
ss.head()

In [ ]:
ss_.head()

In [ ]:
ss

In [ ]:
ss_

In [ ]:
from sklearn.metrics import log_loss

In [ ]:
## Compute the log-loss
loss = log_loss(y_true, y_pred_prob)
print("LogLoss:", loss)

loss_ = log_loss(y_true_, y_pred_prob_)
print("LogLoss:", loss_)

In [ ]:
train_df.columns

In [ ]:
train_df_.columns